# O.G.R.E. Fine-Tuning: Qwen3.5 9B Statistics Grader

Fine-tunes Qwen3.5-9B with LoRA on **415 grading examples** (239 text + 176 vision/handwriting).

**Pipeline:** Install deps → Load model → Upload data (text + vision) → Train → Smoke test → Export GGUF → Download

**Requirements:** Google Colab with **A100 GPU** (Colab Pro). L4 works with 4-bit fallback. T4 is too small.

**Crash recovery:** Checkpoints auto-save to Google Drive. If disconnected, re-run cells 1-3b then Cell 4 auto-recovers.

---

## Steps

1. **Cell 1:** Install dependencies
2. **Cell 2:** Load full Qwen3.5-9B in bf16 with LoRA (4-bit fallback on non-A100 GPUs)
3. **Cell 3:** Upload `finetune-grading.jsonl` (239 text examples)
4. **Cell 3b:** Upload `finetune-grading-vision.jsonl` (176 vision examples)
5. Upload `finetune-grading-val.jsonl` AND `finetune-grading-val-vision.jsonl` to Colab file browser before Cell 4
6. **Cell 4:** Train (auto-recovers from Drive checkpoint if runtime disconnected)
7. **Cell 4b:** Post-training smoke test (MUST pass before proceeding)
8. **Cell 5:** Export to GGUF Q4_K_M
9. **Cell 5b:** Copy GGUF to Google Drive + MD5 checksum
10. **Cell 6:** Download GGUF
11. **Cell 7:** Push to Hugging Face (optional)

After download, run locally:
```bash
ollama create qwen3.5-9B-stat-grader -f fine-tuned-model/Modelfile-qwen3.5-9B-stat-grader
```

## 1. Install Dependencies

Installs Unsloth, Transformers v5, TRL, and other training dependencies. Also checks GPU type and VRAM.

**~3 minutes** on a fresh runtime.

In [ ]:
import subprocess, sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--upgrade",
        "unsloth",
        "unsloth_zoo",
        "Pillow<10",  # Pin early — Pillow 10+ removed is_directory from PIL._util
    ],
    check=True,
)

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "transformers>=5.2.0",  # Qwen3.5 requires transformers v5
        "trl>=0.15.0",
        "datasets",
        "bitsandbytes",
        "sentencepiece",
        "protobuf",
    ],
    check=True,
)

# Re-pin Pillow — later deps may override the constraint (Pillow 10+ breaks is_directory)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "pillow<10", "-q"],
    check=True,
)

import torch

gpu_name = torch.cuda.get_device_name(0)
gpu_vram_gb = round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1)
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {gpu_name}")
print(f"VRAM: {gpu_vram_gb} GB")

use_4bit = "A100" not in gpu_name
if use_4bit:
    print(f"⚠️  Expected A100 but got {gpu_name} — falling back to 4-bit quantization")
else:
    print("✅ A100 confirmed — using bf16 LoRA (best quality)")

## 2. Load Qwen3.5-9B with LoRA

Downloads the base model and attaches LoRA adapters.

- `max_seq_length=8192`
- LoRA rank 16, alpha 16
- Vision encoder frozen, language layers trained
- Auto-detects A100 (bf16) vs other GPUs (4-bit fallback)

In [ ]:
import os
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"  # Disable unsloth compilation (Qwen3.5 rotary crash)

from unsloth import FastVisionModel

MAX_SEQ_LENGTH = 8192
LORA_RANK = 16
LORA_ALPHA = 16

model, tokenizer = FastVisionModel.from_pretrained(
    model_name="Qwen/Qwen3.5-9B",
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=use_4bit,
    load_in_16bit=not use_4bit,
    dtype=torch.bfloat16,
)

model = FastVisionModel.get_peft_model(
    model,
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    bias="none",
    # Freeze vision encoder — preserves image understanding without retraining it
    finetune_vision_layers=False,
    # Fine-tune language layers only — this is where grading behavior lives
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print("Model loaded. Trainable params:")
model.print_trainable_parameters()

## 3. Upload Training Data (Text)

Upload `finetune-grading.jsonl` when prompted (239 text grading examples).

Verifies system/user/assistant message format and converts to HuggingFace Dataset with Qwen3.5 chat template (`enable_thinking=False`).

In [ ]:
from google.colab import files as colab_files
import json

print("Upload finetune-grading.jsonl when prompted...")
uploaded = colab_files.upload()  # select finetune-grading.jsonl from your machine

# Parse JSONL
filename = list(uploaded.keys())[0]
raw_lines = uploaded[filename].decode("utf-8").strip().split("\n")
examples = [json.loads(l) for l in raw_lines if l.strip()]
print(f"Loaded {len(examples)} training examples")

# Verify format
assert examples[0]["messages"][0]["role"] == "system", "msg[0] should be system"
assert examples[0]["messages"][1]["role"] == "user", "msg[1] should be user"
assert examples[0]["messages"][2]["role"] == "assistant", "msg[2] should be assistant"
print("Format check passed")

# Convert to HuggingFace dataset
from datasets import Dataset


def apply_chat_template(example):
    """Format messages using Qwen3.5 chat template — thinking OFF."""
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
        enable_thinking=False,  # NO thinking mode — plan constraint
    )
    return {"text": text}


dataset = Dataset.from_list(examples)
dataset = dataset.map(apply_chat_template, batched=False)
print(f"Dataset ready. Sample:\n{dataset[0]['text'][:300]}...")

## 3b. Upload Training Data (Vision)

**REQUIRED.** Upload `finetune-grading-vision.jsonl` (176 handwriting image examples).

Decodes base64 PNG images, embeds them in messages, and builds combined training list.

**Combined: 415 total training examples** (239 text + 176 vision).

In [ ]:
import base64, io, random as _random
from PIL import Image as _PILImage
from transformers import AutoProcessor

has_vision = False  # updated to True if vision data loads successfully
_processor = None  # initialized here so it's always bound
_collator = None

try:
    print("Upload finetune-grading-vision.jsonl when prompted...")
    _uv = colab_files.upload()
    _vf = list(_uv.keys())[0]
    _vlines = _uv[_vf].decode("utf-8").strip().split("\n")
    _vraw = [json.loads(l) for l in _vlines if l.strip()]
    print(f"Loaded {len(_vraw)} vision examples from {_vf}")

    # Load the vision processor (needed for collator + inference)
    print("Loading vision processor...")
    _processor = AutoProcessor.from_pretrained(
        "Qwen/Qwen3.5-9B",
        trust_remote_code=True,
    )

    # Build unified training list (unsloth format — plain Python dicts, no Arrow)
    # Helper: convert string content to list format (unsloth expects list content always)
    def _to_list_content(msgs):
        out = []
        for m in msgs:
            m = dict(m)
            if isinstance(m["content"], str):
                m["content"] = [{"type": "text", "text": m["content"]}]
            out.append(m)
        return out

    # Text examples — convert string content to list format
    training_data = [{"messages": _to_list_content(_ex["messages"])} for _ex in examples]
    _n_text = len(training_data)

    # Vision examples — decode images, embed in user message content
    for _idx, _ex in enumerate(_vraw):
        _msgs = _ex["messages"]

        # Decode images from data URI
        _pil_images = []
        for _img_str in _ex.get("images", []):
            _b64 = _img_str.split(",", 1)[1] if "," in _img_str else _img_str
            _pil_images.append(
                _PILImage.open(io.BytesIO(base64.b64decode(_b64))).convert("RGB")
            )

        # Convert all messages to list content + fill image placeholders
        _conv_msgs = []
        for _m in _msgs:
            _m_copy = dict(_m)
            # All content must be list format (unsloth requirement)
            if isinstance(_m_copy["content"], str):
                _m_copy["content"] = [{"type": "text", "text": _m_copy["content"]}]
            else:
                _m_copy["content"] = [dict(c) for c in _m_copy["content"]]
            # Fill {"type":"image"} placeholders with actual PIL images
            _img_idx = 0
            for _part in _m_copy["content"]:
                if _part.get("type") == "image" and _img_idx < len(_pil_images):
                    _part["image"] = _pil_images[_img_idx]
                    _img_idx += 1
            _conv_msgs.append(_m_copy)

        training_data.append({"messages": _conv_msgs})

        if (_idx + 1) % 8 == 0:
            print(f"  Processed {_idx + 1}/{len(_vraw)} vision examples...")

    _random.Random(42).shuffle(training_data)
    has_vision = True

    print(
        f"Training data: {len(training_data)} total ({_n_text} text + {len(_vraw)} vision)"
    )

except (KeyboardInterrupt, Exception) as _e:
    raise RuntimeError(
        "finetune-grading-vision.jsonl is required for this 459-example training run"
    ) from _e

## 4. Train

**~45-60 min on A100.** Mounts Google Drive for checkpoint persistence.

- 2 epochs, lr=2e-5, cosine schedule, warmup 10%
- Checkpoints saved every 50 steps + copied to Drive
- **Crash recovery:** If a Drive checkpoint exists, training is skipped and the model is restored

> **Before running:** Upload `finetune-grading-val.jsonl` to the Colab file browser (left sidebar). Do NOT use the upload dialog — drag the file directly.

In [ ]:
import os, glob, shutil
from google.colab import drive
from peft import PeftModel
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

CHECKPOINT_DIR = "/content/outputs_qwen35_grader"
DRIVE_CHECKPOINT_DIR = "/content/drive/MyDrive/qwen35-grader-checkpoint"

# Mount Drive — used for both saving and recovery
drive.mount("/content/drive")

# Vision data collator — used when training includes image examples
if has_vision:
    from unsloth.trainer import UnslothVisionDataCollator

    _collator = UnslothVisionDataCollator(model, _processor)
    print("Vision data collator enabled.")

# Check if a completed checkpoint exists on Drive (runtime recovery)
_ckpt_dirs = glob.glob(f"{DRIVE_CHECKPOINT_DIR}/checkpoint-*")
drive_checkpoints = sorted(_ckpt_dirs, key=lambda p: int(p.rsplit("-", 1)[-1]))
if drive_checkpoints:
    latest = drive_checkpoints[-1]
    print(f"Found Drive checkpoint: {latest}")
    print("Skipping training — loading from checkpoint instead.")
    # Load adapter weights into existing LoRA model (Cell 2 already applied adapters)
    model.load_adapter(latest, adapter_name="default")
    print("Model restored. Proceed to Cell 5.")
else:
    # No checkpoint — train from scratch
    # NOTE: Upload finetune-grading-val.jsonl AND finetune-grading-val-vision.jsonl
    #       to Colab file browser before running this cell
    import json as _json

    # Text validation — convert string content to list format (same as training)
    _val_lines = open("finetune-grading-val.jsonl").read().strip().split("\n")
    _val_raw = [_json.loads(l) for l in _val_lines if l.strip()]
    val_data = [{"messages": _to_list_content(_vex["messages"])} for _vex in _val_raw]
    print(f"Text validation: {len(val_data)} examples")

    # Vision validation — decode images, embed in messages
    if has_vision and os.path.exists("finetune-grading-val-vision.jsonl"):
        _val_vision_lines = (
            open("finetune-grading-val-vision.jsonl").read().strip().split("\n")
        )
        _val_vraw = [_json.loads(l) for l in _val_vision_lines if l.strip()]
        for _vex in _val_vraw:
            _vmsg = _vex["messages"]
            _vpil_imgs = []
            for _vimg_str in _vex.get("images", []):
                _vb64 = _vimg_str.split(",", 1)[1] if "," in _vimg_str else _vimg_str
                _vpil_imgs.append(
                    _PILImage.open(io.BytesIO(base64.b64decode(_vb64))).convert("RGB")
                )
            _conv_msgs = []
            for _m in _vmsg:
                _m_copy = dict(_m)
                if isinstance(_m_copy["content"], str):
                    _m_copy["content"] = [{"type": "text", "text": _m_copy["content"]}]
                else:
                    _m_copy["content"] = [dict(c) for c in _m_copy["content"]]
                _img_idx = 0
                for _part in _m_copy["content"]:
                    if _part.get("type") == "image" and _img_idx < len(_vpil_imgs):
                        _part["image"] = _vpil_imgs[_img_idx]
                        _img_idx += 1
                _conv_msgs.append(_m_copy)
            val_data.append({"messages": _conv_msgs})
        print(f"Vision validation: {len(_val_vraw)} examples")
    else:
        print("No vision validation file found — validating on text only")

    print(f"Combined validation: {len(val_data)} examples")

    _sft_args = SFTConfig(
        dataset_text_field="",  # collator handles formatting from messages
        dataset_kwargs={"skip_prepare_dataset": True},  # plain list, no HF Dataset
        max_seq_length=MAX_SEQ_LENGTH,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        num_train_epochs=1,  # 1 epoch — Qwen3.5 rotary bug crashes at epoch 2 boundary
        learning_rate=2e-5,
        warmup_ratio=0.1,
        lr_scheduler_type="cosine",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        optim="adamw_8bit",
        logging_steps=5,
        eval_strategy="epoch",
        save_total_limit=3,
        save_steps=50,
        output_dir=CHECKPOINT_DIR,
        seed=42,
        dataset_num_proc=1,
        report_to="none",  # disable wandb
        remove_unused_columns=False,
    )

    _trainer_kwargs = dict(
        model=model,
        tokenizer=tokenizer,
        train_dataset=training_data,
        eval_dataset=val_data,
        args=_sft_args,
    )
    if has_vision:
        # Vision collator handles image encoding and mixed text+vision batches
        _trainer_kwargs["data_collator"] = _collator

    trainer = SFTTrainer(**_trainer_kwargs)

    # Train on responses only — model learns assistant outputs, not prompts
    # (skip for vision training — train_on_responses_only doesn't support image tokens)
    if not has_vision:
        trainer = train_on_responses_only(
            trainer,
            instruction_part="<|im_start|>user\n",
            response_part="<|im_start|>assistant\n",
        )

    print("Starting training...")
    try:
        train_result = trainer.train()
        print("Training complete!")
        print(f"  Final loss: {train_result.training_loss:.4f}")
        print(f"  Steps: {train_result.global_step}")
    finally:
        # Save to Google Drive on completion OR crash/disconnect —
        # preserves the last save_steps=50 checkpoint either way
        _ckpt_list = glob.glob(f"{CHECKPOINT_DIR}/checkpoint-*")
        checkpoints = sorted(_ckpt_list, key=lambda p: int(p.rsplit("-", 1)[-1]))
        if checkpoints:
            print(f"Saving checkpoint to Drive: {DRIVE_CHECKPOINT_DIR}")
            shutil.copytree(CHECKPOINT_DIR, DRIVE_CHECKPOINT_DIR, dirs_exist_ok=True)
            print(f"Checkpoint saved to Drive ({len(checkpoints)} checkpoint(s)).")
        else:
            print(
                "No checkpoints found to save — training may have crashed before step 50."
            )

## 4b. Post-Training Smoke Test

**MANDATORY before GGUF export.** Runs test prompts through the trained model to verify it returns valid JSON with scores.

All tests must pass before proceeding.

In [ ]:
import json as _smoke_json
import re as _smoke_re

FastVisionModel.for_inference(model)


def _extract_json(text):
    """Extract first JSON object from text, even if preceded by thinking output."""
    # Try the whole string first
    text = text.strip()
    try:
        return _smoke_json.loads(text)
    except _smoke_json.JSONDecodeError:
        pass
    # Find first { ... } block
    match = _smoke_re.search(r"\{[^{}]*\}", text)
    if match:
        return _smoke_json.loads(match.group())
    raise ValueError(f"No JSON object found in output: {text[:200]}")


# All content must be list format — processor iterates content looking for images
_smoke_prompts = [
    {
        "name": "Text-only grading (stats)",
        "messages": [
            {"role": "system", "content": [{"type": "text", "text": "You are an expert grading assistant."}]},
            {
                "role": "user",
                "content": [{"type": "text", "text": (
                    "Grade this student response on a scale of 0-10.\n\n"
                    "RUBRIC: Explain what a p-value represents in hypothesis testing.\n\n"
                    "STUDENT RESPONSE: A p-value is the probability that the null hypothesis is true. "
                    "If p < 0.05 we reject it.\n\n"
                    'Return JSON: {"score": <0-10>, "feedback": "..."}'
                )}],
            },
        ],
    },
    {
        "name": "Text-only grading (different rubric)",
        "messages": [
            {"role": "system", "content": [{"type": "text", "text": "You are an expert grading assistant."}]},
            {
                "role": "user",
                "content": [{"type": "text", "text": (
                    "Grade this student response on a scale of 0-10.\n\n"
                    "RUBRIC: Describe the Central Limit Theorem and its importance.\n\n"
                    "STUDENT RESPONSE: The CLT says that sample means are normally distributed "
                    "when n is large enough, regardless of the population distribution. "
                    "This lets us use z-tests and confidence intervals.\n\n"
                    'Return JSON: {"score": <0-10>, "feedback": "..."}'
                )}],
            },
        ],
    },
]

print("=== POST-TRAINING SMOKE TEST ===")
_smoke_pass = 0
for _sp in _smoke_prompts:
    print(f"\n--- {_sp['name']} ---")
    _text = _processor.apply_chat_template(
        _sp["messages"],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    _inputs = _processor(text=[_text], return_tensors="pt").to(model.device)
    _output = model.generate(**_inputs, max_new_tokens=512, temperature=0.2)
    _decoded = tokenizer.decode(
        _output[0][_inputs["input_ids"].shape[-1] :], skip_special_tokens=True
    )
    print(f"Output: {_decoded[:500]}")
    try:
        _parsed = _extract_json(_decoded)
        _score = _parsed.get("score")
        assert isinstance(_score, (int, float)) and 0 <= _score <= 10, f"score={_score}"
        print(f"✅ Valid JSON, score={_score}")
        _smoke_pass += 1
    except Exception as _e:
        print(f"❌ FAILED: {_e}")

# Vision smoke test — use first image from training data if available
if has_vision and _vraw:
    print(f"\n--- Vision grading (handwriting image) ---")
    _vex = _vraw[0]
    _vmsg = _vex["messages"]
    _vimg_str = _vex["images"][0]
    _vb64 = _vimg_str.split(",", 1)[1] if "," in _vimg_str else _vimg_str
    _vpil = _PILImage.open(io.BytesIO(base64.b64decode(_vb64))).convert("RGB")

    _vtext = _processor.apply_chat_template(
        _vmsg[:2],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    _vinputs = _processor(text=[_vtext], images=[_vpil], return_tensors="pt").to(
        model.device
    )
    _voutput = model.generate(**_vinputs, max_new_tokens=512, temperature=0.2)
    _vdecoded = tokenizer.decode(
        _voutput[0][_vinputs["input_ids"].shape[-1] :], skip_special_tokens=True
    )
    print(f"Output: {_vdecoded[:500]}")
    try:
        _vparsed = _extract_json(_vdecoded)
        assert "score" in _vparsed, "missing 'score'"
        assert "transcription" in _vparsed, "missing 'transcription'"
        print(
            f"✅ Valid JSON, score={_vparsed['score']}, transcription length={len(_vparsed['transcription'])}"
        )
        _smoke_pass += 1
    except Exception as _e:
        print(f"❌ FAILED: {_e}")

_smoke_total = len(_smoke_prompts) + (1 if has_vision else 0)
print(f"\n=== SMOKE TEST RESULT: {_smoke_pass}/{_smoke_total} passed ===")
if _smoke_pass < _smoke_total:
    print("⚠️  Some smoke tests failed. Review output before proceeding to GGUF export.")
else:
    print("✅ All smoke tests passed — safe to proceed to Cell 5 (GGUF export).")

## 5. Export to GGUF Q4_K_M

Exports the fine-tuned model to GGUF format (Q4_K_M quantization) for use with Ollama.

**~10-15 minutes.** Output: ~5-6 GB file.

In [ ]:
GGUF_OUTPUT_DIR = "/content/qwen35-stat-grader-gguf"
GGUF_ACTUAL_DIR = "/content/qwen35-stat-grader-gguf_gguf"  # Unsloth appends _gguf

print("Exporting to GGUF Q4_K_M...")
model.save_pretrained_gguf(
    GGUF_OUTPUT_DIR,
    tokenizer,
    quantization_method="q4_k_m",
)

import os, glob

# Unsloth saves to <dir>_gguf — search both locations
gguf_files = glob.glob(f"{GGUF_ACTUAL_DIR}/*.Q4_K_M.gguf") or glob.glob(
    f"{GGUF_OUTPUT_DIR}/*.gguf"
)
print(f"GGUF files found: {gguf_files}")
for f in gguf_files:
    size_gb = os.path.getsize(f) / 1024**3
    print(f"  {f}: {size_gb:.2f} GB")

## 5b. Save GGUF to Google Drive

Copies GGUF to Google Drive for persistent storage (Colab's `/content/` is ephemeral) and generates an MD5 checksum.

**Record the MD5 checksum** — you'll verify the download later.

In [ ]:
import shutil, hashlib, os

gguf_files = glob.glob(f"{GGUF_ACTUAL_DIR}/*.Q4_K_M.gguf") or glob.glob(
    f"{GGUF_OUTPUT_DIR}/*.gguf"
)
assert gguf_files, "No GGUF file found — run Cell 5 first"
gguf_path = gguf_files[0]
print(f"Found GGUF: {gguf_path} ({os.path.getsize(gguf_path) / 1024**3:.2f} GB)")

DRIVE_GGUF_DIR = "/content/drive/MyDrive/models"
DRIVE_GGUF_NAME = "qwen3.5-9B-stat-grader-Q4_K_M.gguf"
os.makedirs(DRIVE_GGUF_DIR, exist_ok=True)
drive_gguf_path = f"{DRIVE_GGUF_DIR}/{DRIVE_GGUF_NAME}"

print(f"Copying GGUF to Drive: {drive_gguf_path}")
print("This may take 2-5 minutes...")
shutil.copy(gguf_path, drive_gguf_path)
print("Copy complete!")

print("Generating md5 checksum...")
md5 = hashlib.md5()
with open(drive_gguf_path, "rb") as f:
    for chunk in iter(lambda: f.read(8192), b""):
        md5.update(chunk)
checksum = md5.hexdigest()
print(f"MD5: {checksum}")

checksum_path = f"{DRIVE_GGUF_DIR}/{DRIVE_GGUF_NAME}.md5"
with open(checksum_path, "w") as f:
    f.write(f"{checksum}  {DRIVE_GGUF_NAME}\n")
print(f"Checksum saved to Drive: {checksum_path}")

print(f"""
=== GGUF Saved to Drive ===
File: {drive_gguf_path}
MD5:  {checksum}

IMPORTANT: Record this MD5 checksum!
After downloading locally, verify with:
  md5sum fine-tuned-model/qwen3.5-9B-stat-grader-Q4_K_M.gguf
Expected: {checksum}
""")

## 6. Download GGUF

Downloads the GGUF file to your local machine.

**Recommended:** Download from [drive.google.com](https://drive.google.com) → My Drive → models/ (more reliable for large files).

**Fallback:** Browser download from Colab (unreliable for 5+ GB files).

In [ ]:
# PRIMARY METHOD: Download from Google Drive (more reliable for large files)
#   1. Go to drive.google.com
#   2. Navigate to My Drive > models/
#   3. Download: qwen3.5-9B-stat-grader-Q4_K_M.gguf (~5-6 GB)
#   4. Place in: O.G.R.E-OllamaGradingRubricEvaluator/fine-tuned-model/
#
# FALLBACK METHOD: Direct Colab download (unreliable for large files)

import os
from google.colab import files as colab_files

# Unsloth saves to <dir>_gguf — search both locations
gguf_files = glob.glob(f"{GGUF_ACTUAL_DIR}/*.Q4_K_M.gguf") or glob.glob(
    f"{GGUF_OUTPUT_DIR}/*.gguf"
)
assert gguf_files, "No GGUF found — run Cell 5 first"
gguf_path = gguf_files[0]
size_gb = os.path.getsize(gguf_path) / 1024**3
print(f"Attempting browser download: {gguf_path} ({size_gb:.2f} GB)")
print("NOTE: For reliable download, use Google Drive (see comments above)")
colab_files.download(gguf_path)

print("""
=== DONE — Next Steps (run locally after download) ===

1. Move the downloaded .gguf to:
   fine-tuned-model/qwen3.5-9B-stat-grader-Q4_K_M.gguf

2. Verify integrity (compare with MD5 from Cell 5b):
   md5sum fine-tuned-model/qwen3.5-9B-stat-grader-Q4_K_M.gguf

3. Register in Ollama (run from fine-tuned-model/ directory):
   ollama create qwen3.5-9B-stat-grader -f Modelfile-qwen3.5-9B-stat-grader

4. Verify:
   ollama run qwen3.5-9B-stat-grader "test"

5. Run benchmark:
   bun run test-data/run-benchmark.js --only=qwen3.5-9B-stat-grader
""")

## 7. Push to Hugging Face (Optional)

Pushes LoRA adapters to Hugging Face Hub for reproducibility.

**Prerequisites:**
1. Create a free account at [huggingface.co](https://huggingface.co)
2. Create a write-access token at [Settings → Tokens](https://huggingface.co/settings/tokens)

In [ ]:
# Prerequisites:
#   1. Create a free account at https://huggingface.co
#   2. Create a write-access token at https://huggingface.co/settings/tokens
#   3. Create a repo at https://huggingface.co/new (or let this script create it)
#
# Set your HF username and desired repo name below, then run this cell.
# You will be prompted to paste your token when huggingface_hub.login() runs.

HF_USERNAME = "shuff57"
HF_REPO_NAME = "qwen3.5-math-grader"

# ── Login ────────────────────────────────────────────────────────────────────
from huggingface_hub import login, HfApi

login()  # prompts for token; paste your write-access HF token

# ── Option A: Push LoRA adapters only (~100 MB, fast upload) ─────────────────
# Useful for reproducibility — anyone with the base model can merge these.
print(f"\nPushing LoRA adapters to {HF_USERNAME}/{HF_REPO_NAME}-lora ...")
model.push_to_hub(
    f"{HF_USERNAME}/{HF_REPO_NAME}-lora",
    tokenizer=tokenizer,
    private=False,  # set True if you want a private repo
)
tokenizer.push_to_hub(f"{HF_USERNAME}/{HF_REPO_NAME}-lora", private=False)
print("LoRA adapters pushed.")

# ── Option B: Push GGUF directly (~5 GB, slower upload) ──────────────────────
# Useful so others can ollama pull directly from HF without any merge step.
# Comment out Option A above and uncomment below to use instead (or run both).

# print(f"\nPushing GGUF to {HF_USERNAME}/{HF_REPO_NAME}-gguf ...")
# model.push_to_hub_gguf(
#     f"{HF_USERNAME}/{HF_REPO_NAME}-gguf",
#     tokenizer,
#     quantization_method="q4_k_m",
#     private=False,
# )
# print("GGUF pushed.")

# ── Print usage instructions ──────────────────────────────────────────────────
print(f"""
=== Pushed to Hugging Face ===

LoRA adapters: https://huggingface.co/{HF_USERNAME}/{HF_REPO_NAME}-lora

To use the GGUF in Ollama from HF:
  1. Download the .gguf from the HF repo Files tab
  2. Place in fine-tuned-model/qwen3.5-math-grader.gguf
  3. ollama create qwen3.5-math-grader -f fine-tuned-model/Modelfile-qwen3.5-math-grader
""")

---

## Next Steps (Local Machine)

After downloading the GGUF file:

```bash
# 1. Move GGUF to the project
mv ~/Downloads/qwen3.5-9B-stat-grader-Q4_K_M.gguf fine-tuned-model/

# 2. Verify integrity (compare MD5 with Cell 5b output)
md5sum fine-tuned-model/qwen3.5-9B-stat-grader-Q4_K_M.gguf

# 3. Create Ollama model
cd fine-tuned-model
ollama create qwen3.5-9B-stat-grader -f Modelfile-qwen3.5-9B-stat-grader

# 4. Test
ollama run qwen3.5-9B-stat-grader "test"

# 5. Run benchmark (from project root)
bun run test-data/run-benchmark.js --only=qwen3.5-9B-stat-grader
```

The Modelfile is already in `fine-tuned-model/Modelfile-qwen3.5-9B-stat-grader`.